In [10]:
from helpers.persistence import save_var, load_var
from helpers.progress_bar import ProgressBar
from helpers.openml_data_v2 import get_data1, openml_cc18_list, hard_list
from helpers.openml_data import tabular_id_list

In [11]:
import numpy as np
from tqdm import tqdm
import scipy

In [12]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import pairwise_distances, pairwise_distances_chunked

In [13]:
from NPT.run import main
from NPT.npt.configs import build_parser

In [14]:
# npt uses sklearn to do CV and splitting and uses the same random state = 42 and has same test_index
# problem is that somewhere in trainer or deeper the data rows are shuffled and that is the y_preds assertion fails

In [15]:
# dataset_name = 23

# parser = build_parser()
# args = parser.parse_args([
#     '--data_set', f'custom__{dataset_name}', 
#     '--custom_data_set', f'{dataset_name}', 
#     '--exp_test_perc', '0.2',
#     '--exp_val_perc', '0.1',
#     '--exp_patience', '30',
#     '--exp_n_runs', '1',
#     '--exp_num_total_steps', '10',
#     '--exp_batch_size', '128',
#     # '--exp_disable_cuda',
#     # '--data_set_on_cuda', 'True',
#     '--exp_full_batch_gd',
# ])


# fold_preds, fold_trues = main(args)
# # dataset, _ = main(args)
# # args

In [16]:
def get_cv_results(test_preds, test_trues):
    
    scores = []
    y_trues = []
    y_preds = []
    
    
    # pbar.add_prefix('starting 10-fold cv')

    for fold_index, (y_test, y_pred) in enumerate(zip(test_preds, test_trues)):    
        
        score = f1_score(y_test, y_pred, average='weighted')
        
        # acc = ((y_test == y_pred).sum() / y_test.shape[0])
        # print('acc', acc)
        
        scores.append(score)
        y_preds.append(y_pred)
        y_trues.append(y_test)
        
    return scores, y_preds, y_trues


# scores, _, _ = get_cv_results(fold_preds, fold_trues)
# scores

In [17]:
save_path = './saved_vars/test-time-npt.pkl'
dataset_results = load_var(save_path) or {}

cache_path = './saved_vars/test-time-npt-outputs.pkl'
outputs = load_var(cache_path) or {}

dataset_results, outputs = {}, {}

In [18]:
from time import time
start_time, end_time, time_taken = 0, 0, 0

In [19]:
# pbar = ProgressBar(data_loaders.items())
pbar = hard_list
pbar = [ 1063,  1510,  1464,   469,   458,  1494,  1068,  1049,    23,
        1050, 40975, 40982,  1067,  1487,  1485,  4134, 40701,  1497,
        1475,  4538]

pbar = [ 458, 1050, 1475, 1485, 1487, 1497, 4134, 4538][::2]

skip_list = [1050, 1487] #outOfMemory error

# pbar = [1485]
pbar = [1510]

pbar = [i for i in pbar if i not in [458]]
# pbar = [('breast cancer', data_loaders['breast cancer'])]
failed_list = []

for dataset_name in pbar:
    
    # if dataset_name in skip_list:
    #     continue
    
    try:
        df, y, cat = get_data1(dataset_name)
    except Exception as e:
        print('couldnt do', dataset_name)
        # raise e
        failed_list.append(dataset_name)
        continue
        
    if df.shape[1] > 100:
        continue
        
    batch_size = 128
    # NPT takes max steps as input and calculates epochs from there.
    # NPT takes full dataset, breaks into batch size which is weird.
    # normaly full dataset is split into train, valid, test. Train set is broken into mini-batches
    n_batches = int(np.ceil(df.shape[0]/batch_size)) if batch_size > 0 else 1
    n_steps = n_batches * 1000
    # print(n_steps)

    parser = build_parser()
    args = parser.parse_args([
        '--data_set', f'custom__{dataset_name}', 
        '--custom_data_set', f'{dataset_name}', 
        '--exp_test_perc', '0.2',
        '--exp_val_perc', '0.1',
        '--exp_patience', '-1',
        # '--exp_n_runs', '30',
        '--exp_n_runs', '10',
        '--exp_bootstrap', '30',
        '--exp_num_total_steps', f'{n_steps}',
        '--exp_batch_size', f'{batch_size}',
        # '--exp_disable_cuda',
        # '--verbose',
    ])
    
    print('Trying ', dataset_name)
    
    fold_preds, fold_trues = [], []
    
    if dataset_name in outputs.keys():
        fold_preds, fold_trues = outputs[dataset_name]
        print(f'NPT output for {dataset_name} found, skipping model train & eval')
    else:
        start_time = time()
        fold_preds, fold_trues = main(args)
        end_time = time()
        time_taken = end_time - start_time
        print(time_taken)
        outputs[dataset_name] = fold_preds, fold_trues
        save_var(outputs, cache_path)
    
    try:
        if dataset_name in dataset_results.keys():
            print(f'NPT results for {dataset_name} found, skipping loop')
            continue
            
        scores = get_cv_results(fold_preds, fold_trues)
        dataset_results[dataset_name] = scores
        save_var(dataset_results, save_path)

    except Exception as e:
        print('couldnt do', dataset_name)
        # raise e
        failed_list.append(dataset_name)
    
    # print(scores)

2023-08-30 22:25:42 | INFO | openml.datasets.dataset | pickle write wdbc


Trying  1510
Configuring arguments...
Doing k-FOLD CV. Assigning group name wan0ae5n.
Running model with CUDA
data_set_on_cuda False


2023-08-30 22:25:53 | INFO | openml.datasets.dataset | pickle write wdbc


dataset_name 1510
[0]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30] [0]
Percentage of each group: Train 0.70 | 0.10 | 0.20
Percentage of each group: Train 0.70 | 0.10 | 0.20
Percentage of each group: Train 0.70 | 0.10 | 0.20
Percentage of each group: Train 0.70 | 0.10 | 0.20
Percentage of each group: Train 0.70 | 0.10 | 0.20
Percentage of each group: Train 0.70 | 0.10 | 0.20


/home/cida-lab-2/miniconda3/envs/npt/lib/python3.8/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/home/cida-lab-2/miniconda3/envs/npt/lib/python3.8/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/home/cida-lab-2/miniconda3/envs/npt/lib/python3.8/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/home/cida-lab-2/miniconda3/envs/npt/lib/python3.8/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarn

...

Building NPT.
Using feature type embedding (unique embedding for categorical and numerical features).
Using feature index embedding (unique embedding for each column).
Clipping gradients to value 1.0.
Model has 205023200 parameters,batch size 128.
Initialized "lookahead_lamb" optimizer.
Warming up for 3500.0/5000.0 steps.
Initialized "flat_and_anneal" learning rate scheduler.
N 569 ; num_steps_per_epoch 5
Initialized "cosine" augmentation/label tradeoff annealer. Annealing to minimum value in 5000 steps.
Using AUROC in loss module.


/home/cida-lab-2/miniconda3/envs/npt/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:138: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Validation loss has improved 1 times since last caching the model. Caching now.
Storing new best performing model.
Model checkpointing attempts: 0.
Stored epoch 5 model checkpoint to NPT/data/custom__1510/ssl__True/np_seed=42__n_cv_splits=30__exp_num_runs=10__cv_9/model_checkpoints/model_5.pt.
Val loss: 0.2999608516693115.
Validation loss has improved 1 times since last caching the model. Caching now.
Storing new best performing model.
Model checkpointing attempts: 0.
Stored epoch 10 model checkpoint to NPT/data/custom__1510/ssl__True/np_seed=42__n_cv_splits=30__exp_num_runs=10__cv_9/model_checkpoints/model_10.pt.
Val loss: 0.22095359861850739.
Validation loss has improved 1 times since last caching the model. Caching now.
Storing new best performing model.
Model checkpointing attempts: 0.
Stored epoch 15 model checkpoint to NPT/data/custom__1510/ssl__True/np_seed=42__n_cv_splits=30__exp_num_runs=10__cv_9/model_checkpoints/model_15.pt.
Val loss: 0.13016708195209503.
Validation loss has

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
lr,██████████████████████████████▇▇▆▅▄▃▂▂▁▁
step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
test_label_auroc,▁▇█▇▇█▇▇▆▆▇▆▅▆▄▂▇▅▄▄▄▆▂▆▆▅▄▄▄▃▇▄▅▄▅▆▄▅▅▅
test_label_cat_accuracy,▁▆▆▆▇▆▄▇▆▇▆▇▆█▆▆▅▅▆▆▇▅▆▇▆▆▆▆▆▆█▆▇▇▆▆▇▇▇▇
test_label_cat_loss,█▁▃▂▂▂▄▁▂▂▁▂▃▂▃▅▃▅▅▄▄▃▅▂▄▃▃▅▃▆▂▄▄▃▂▄▂▂▂▃
test_label_total_loss,█▁▃▂▂▂▄▁▂▂▁▂▃▂▃▅▃▅▅▄▄▃▅▂▄▃▃▅▃▆▂▄▄▃▂▄▂▂▂▃
test_label_total_loss_unstd,█▁▃▂▂▂▄▁▂▂▁▂▃▂▃▅▃▅▅▄▄▃▅▂▄▃▃▅▃▆▂▄▄▃▂▄▂▂▂▃
test_total_loss,▆▁▂▂▁▁▃▁▂▂▁▂▃▂▃▅▃▆▅▄▄▄▆▃▅▄▄█▅█▃▆▆▅▄▆▄▄▄▅
tradeoff,███████▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁
train_augmentation_num_loss,█▅▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁


9326.212692260742


In [20]:
time_taken/10 #7479.323241472244 / 30 = 747.9323241472244

932.6212692260742